In [30]:
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
from typing import List, Dict, Any
import warnings
warnings.filterwarnings('ignore')

from langchain.llms import OpenAI
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from langchain.chains import LLMChain
from langchain.schema import BaseOutputParser
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

from dotenv import load_dotenv
load_dotenv()

False

In [ ]:
import os
from langchain.chat_models import ChatOpenAI

os.environ["OPENAI_API_KEY"] = ""  # Replace with your OpenRouter API key
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

llm = ChatOpenAI(
    model="qwen/qwen-2.5-72b-instruct",
    temperature=0.3,
    max_tokens=4000,
    openai_api_key=os.environ.get("OPENAI_API_KEY"),
    openai_api_base=os.environ.get("OPENAI_API_BASE")
)

print("✅ OpenRouter LLM configured successfully!")
print(f"🤖 Using model: {llm.model_name}")

✅ OpenRouter LLM configured successfully!
🤖 Using model: qwen/qwen-2.5-72b-instruct


In [32]:
def load_pr_data(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return data
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return []

data_paths = {
    'lodash': r'PR data for Developers\lodash_pr_data_train.json',
}

all_pr_data = {}
for repo_name, file_path in data_paths.items():
    data = load_pr_data(file_path)
    all_pr_data[repo_name] = data

In [33]:
def analyze_developer_activity(all_pr_data):
    developer_stats = {}
    
    for repo_name, prs in all_pr_data.items():
        for pr in prs:
            author = pr.get('author', {}).get('username', 'unknown')
            if author not in developer_stats:
                developer_stats[author] = {
                    'total_prs': 0,
                    'repos': set(),
                    'pr_details': []
                }
            
            developer_stats[author]['total_prs'] += 1
            developer_stats[author]['repos'].add(repo_name)
            developer_stats[author]['pr_details'].append({
                'repo': repo_name,
                'title': pr.get('title', ''),
                'description': pr.get('description', ''),
                'labels': pr.get('labels', []),
                'state': pr.get('state', ''),
                'additions': pr.get('additions', 0),
                'deletions': pr.get('deletions', 0),
                'changed_files_count': pr.get('changed_files_count', 0),
                'language_stats': pr.get('language_stats', {}),
                'created_at': pr.get('created_at', '')
            })
    
    for author in developer_stats:
        developer_stats[author]['repos'] = list(developer_stats[author]['repos'])
    
    return developer_stats

developer_stats = analyze_developer_activity(all_pr_data)

In [34]:
class DeveloperProfile(BaseModel):
    developer_name: str = Field(description="Developer's username")
    experience_level: str = Field(description="Estimated experience level (Junior/Mid/Senior/Lead)")
    primary_skills: List[str] = Field(description="List of primary JavaScript technical skills")
    programming_languages: List[str] = Field(description="Programming languages used (JavaScript-focused)")
    summary: str = Field(description="Overall JavaScript developer profile summary")
    javascript_skill_matrix: Dict[str, List[str]] = Field(description="Detailed JavaScript skill assessment with frequency embedded in both category keys and individual skill values (e.g., 'Framework/Library Expertise, frequency: 10': ['Node.js, frequency: 6', 'Express.js, frequency: 4'])")

profile_parser = PydanticOutputParser(pydantic_object=DeveloperProfile)

js_skill_categories = {
    "Framework/Library Expertise": [
        "Frontend Frameworks: React, Angular, Vue.js, Svelte, Ember.js, Backbone.js, Mithril, Preact, jQuery",
        "Backend Frameworks: Node.js, Express.js, Koa.js",
        "State Management: Redux, Context API, Zustand",
        "Testing Libraries: Jest, Mocha, Chai",
        "Static Site Generation/Server-Side Rendering: Next.js, Nuxt.js"
    ],
    "Asynchronous Programming": ["async/await", "Promises", "Callbacks", "Event Loop", "Concurrency Control"],
    "API Design & Consumption": ["RESTful APIs", "GraphQL", "WebSockets", "Authentication & Authorization", "API Error Handling", "Rate Limiting"],
    "Error Handling": ["Try-Catch", "Custom Error Classes", "Error Boundaries"],
    "Frontend Development Skills": ["DOM Manipulation"],
    "Backend Development Skills": ["Server-Side Logic", "Database Integration", "API Endpoints", "Authentication/Authorization", "Middleware", "Microservices", "Caching"],
    "DevOps & Deployment": ["CI/CD Pipelines", "Containerization", "Cloud Providers", "Load Balancing", "Web Servers"],
    "Advanced JavaScript Concepts": ["Closures", "Higher-Order Functions", "Prototypes & Inheritance", "Module Systems", "Event Delegation", "Memory Management"],
    "Functional Programming": ["Immutability", "Pure Functions", "Declarative Programming", "Composition"],
    "Data Structures & Algorithms": ["Data Structures", "Searching and Sorting Algorithms", "Recursion", "Big O Notation"],
    "TypeScript": ["Type Definitions", "Generics", "Type Inference", "Modules & Namespaces", "Type Narrowing"],
    "Progressive Web Apps (PWA)": ["Service Workers", "Web Push Notifications", "Caching Strategies", "App Shell Model", "Manifest File"],
    "Mobile Development with JavaScript": ["React Native", "Ionic", "Cordova/PhoneGap"],
    "Web Performance Optimization": ["Lazy Loading", "Code Splitting", "Minification & Compression", "Critical Rendering Path Optimization", "Preloading & Prefetching"],
    "Event-Driven Architecture": ["Event Emitters", "Pub/Sub Model"],
    "Dependency Management": ["npm/yarn", "Package-lock.json & Yarn.lock"],
    "Graphical Data Visualization": ["D3.js", "Chart.js", "WebGL & Three.js"]
}

# List of valid category names for strict validation
VALID_CATEGORY_NAMES = list(js_skill_categories.keys())

skills_analysis_prompt = PromptTemplate(
    input_variables=["pr_data", "developer_name"],
    template="""Analyze the following GitHub Pull Request data for developer {developer_name} and extract their JavaScript technical skills and expertise:

PR Data:
{pr_data}

Focus specifically on JavaScript-related skills. Based on this PR data, identify:
1. JavaScript frameworks and libraries used (React, Vue, Angular, Node.js, etc.)
2. JavaScript programming patterns and paradigms
3. Frontend vs Backend JavaScript contributions
4. Testing frameworks and methodologies in JavaScript
5. Build tools and development workflow
6. JavaScript ES6+ features usage
7. TypeScript usage and proficiency
8. API development and consumption patterns
9. Asynchronous programming patterns
10. Code quality and modern JavaScript practices

Provide a detailed analysis focusing exclusively on JavaScript ecosystem skills and expertise.
Look for evidence in PR titles, descriptions, file changes, and technology stacks.""",
    partial_variables={"format_instructions": "Provide a detailed JavaScript-focused technical analysis."}
)

profile_generation_prompt = PromptTemplate(
    input_variables=["skills_analysis", "developer_name", "pr_count", "repo_list"],
    template="""Based on the following JavaScript skills analysis, generate a comprehensive JavaScript developer profile:

Developer: {developer_name}
Total PRs: {pr_count}
Repositories: {repo_list}

Skills Analysis:
{skills_analysis}

VALID CATEGORY NAMES (THESE ARE THE ONLY ALLOWED KEYS):
{valid_categories}

JavaScript Skill Matrix Details:
{js_categories}

Generate a JavaScript developer profile that includes:
1. Experience level assessment (Junior/Mid/Senior/Lead) based on JavaScript contributions
2. Primary JavaScript technical skills and frameworks they show expertise in
3. Overall JavaScript developer summary
4. Detailed skill matrix mapping to the provided JavaScript categories WITH EMBEDDED FREQUENCY

========== CRITICAL RULES FOR javascript_skill_matrix ==========
YOU MUST FOLLOW THESE RULES EXACTLY OR THE OUTPUT WILL BE REJECTED:

FREQUENCY EMBEDDING RULES:
1. For each category that has skills, ADD the frequency count to the category name in the KEY
2. Format: "Category Name, frequency: X" where X is the total count for that category
3. For each individual skill, ALSO ADD the frequency count to the skill name in the VALUE
4. Format: "Skill Name, frequency: Y" where Y is the count for that specific skill
5. For categories with NO skills (empty arrays), DO NOT add frequency - just use the category name alone

CATEGORY NAME RULES:
1. ONLY use these EXACT base category names:
   - "Framework/Library Expertise"
   - "Asynchronous Programming"
   - "API Design & Consumption"
   - "Error Handling"
   - "Frontend Development Skills"
   - "Backend Development Skills"
   - "DevOps & Deployment"
   - "Advanced JavaScript Concepts"
   - "Functional Programming"
   - "Data Structures & Algorithms"
   - "TypeScript"
   - "Progressive Web Apps (PWA)"
   - "Mobile Development with JavaScript"
   - "Web Performance Optimization"
   - "Event-Driven Architecture"
   - "Dependency Management"
   - "Graphical Data Visualization"

2. DO NOT and NEVER EVER create any new category names
3. DO NOT and NEVER EVER use subcategory names like "Testing Libraries" or "Backend Frameworks"
4. DO NOT and NEVER EVER use individual skill names like "Node.js", "Express.js", "CI/CD" as category names
5. If you see skills related to testing (Jest, Mocha, etc.), count them under "Framework/Library Expertise"

Example CORRECT javascript_skill_matrix:
{{
  "Framework/Library Expertise, frequency: 15": ["Node.js, frequency: 8", "Express.js, frequency: 5", "Jest, frequency: 2"],
  "Asynchronous Programming, frequency: 6": ["Promises, frequency: 4", "async/await, frequency: 2"],
  "API Design & Consumption, frequency: 10": ["RESTful APIs, frequency: 10"],
  "Backend Development Skills, frequency: 20": ["Middleware, frequency: 8", "API Endpoints, frequency: 7", "Authentication/Authorization, frequency: 5"],
  "DevOps & Deployment, frequency: 8": ["CI/CD Pipelines, frequency: 5", "Containerization, frequency: 3"],
  "Advanced JavaScript Concepts, frequency: 4": ["Prototypes & Inheritance, frequency: 2", "Module Systems, frequency: 2"],
  "Dependency Management, frequency: 12": ["npm/yarn, frequency: 12"],
  "Error Handling": [],
  "Frontend Development Skills": [],
  "Functional Programming": [],
  "Data Structures & Algorithms": [],
  "TypeScript": [],
  "Progressive Web Apps (PWA)": [],
  "Mobile Development with JavaScript": [],
  "Web Performance Optimization": [],
  "Event-Driven Architecture": [],
  "Graphical Data Visualization": []
}}

Example WRONG javascript_skill_matrix (DO NOT DO THIS):
{{
  "Testing Libraries, frequency: 5": ["Jest, frequency: 5"],  ← WRONG! "Testing Libraries" is not a main category
  "Node.js": ["Express.js"],  ← WRONG! "Node.js" is a skill, not a category
  "Framework/Library Expertise": ["Node.js", "Express.js"],  ← WRONG! Missing frequency for non-empty array AND individual skills
  "Framework/Library Expertise, frequency: 10": ["Node.js", "Express.js"]  ← WRONG! Missing frequency for individual skills
}}
========== END OF CRITICAL RULES FOR FREQUENCY EMBEDDING ==========

========== CRITICAL RULES FOR SKILL SELECTION ==========
YOU MUST FOLLOW THESE RULES EXACTLY FOR THE SKILL MATRIX:

1. The javascript_skill_matrix MUST use the EXACT 17 category names as keys
2. For each category, ONLY include skills that are EXPLICITLY LISTED in the subcategory definitions above
3. DO NOT add skills that are not in the predefined lists (e.g., "webpack", "babel", etc.)
4. If no evidence found for a category, use an empty array []
5. DO NOT invent or add new skills - ONLY use skills from the predefined lists

Example CORRECT javascript_skill_matrix:
{{
  "Framework/Library Expertise, frequency: 15": ["Node.js, frequency: 8", "Express.js, frequency: 5", "Jest, frequency: 2"],
  "API Design & Consumption, frequency: 10": ["RESTful APIs, frequency: 8", "Authentication & Authorization, frequency: 2"],
  "Backend Development Skills, frequency: 20": ["Middleware, frequency: 8", "API Endpoints, frequency: 12"],
  "Dependency Management, frequency: 12": ["npm/yarn, frequency: 12"],
  "TypeScript": []
}}

Example WRONG javascript_skill_matrix (DO NOT DO THIS):
{{
  "Framework/Library Expertise": ["Node.js", "Express.js", "webpack"],  ← WRONG! "webpack" not in list AND missing frequencies
  "API Design & Consumption, frequency: 10": ["RESTful APIs", "GraphQL", "custom-api"],  ← WRONG! "custom-api" not in list AND missing individual frequencies
  "DevOps & Deployment": ["CI/CD Pipelines", "Docker", "Kubernetes"]  ← WRONG! "Docker", "Kubernetes" not in list, AND missing frequencies
}}
========== END OF CRITICAL RULES ==========

{format_instructions}

Be specific and evidence-based in your JavaScript skill assessment.
Map identified skills to the appropriate categories in the skill matrix.
Count skill usage frequency based on evidence from PR titles, descriptions, and file changes.
REMEMBER: 
- Embed frequency counts in BOTH the category names AND individual skill names for non-empty skill arrays!
- Only use the 17 exact base category names!
- Only use skills EXPLICITLY listed in the subcategory definitions for javascript_skill_matrix!
- DO NOT invent new skills or add skills not in the predefined lists!
- Format: "Category Name, frequency: X": ["skill1, frequency: Y", "skill2, frequency: Z"]
- The category frequency should be the SUM of all individual skill frequencies in that category
""",
    partial_variables={
        "format_instructions": profile_parser.get_format_instructions(),
        "js_categories": "\n".join([f"{cat}: {', '.join(skills)}" for cat, skills in js_skill_categories.items()]),
        "valid_categories": "\n".join([f"   - \"{cat}\"" for cat in VALID_CATEGORY_NAMES])
    }
)

In [35]:
def create_analysis_chains(llm):
    
    skills_chain = LLMChain(
        llm=llm,
        prompt=skills_analysis_prompt,
        verbose=True
    )
    
    profile_chain = LLMChain(
        llm=llm,
        prompt=profile_generation_prompt,
        verbose=True
    )
    
    return skills_chain, profile_chain

def prepare_pr_data_for_analysis(developer_data, max_prs=20):
    """Prepare PR data for LLM analysis (limit size to avoid token limits)"""
    
    pr_details = developer_data['pr_details'][:max_prs] 
    
    pr_summary = []
    for pr in pr_details:
        summary = {
            'title': pr['title'][:100],
            'repo': pr['repo'],
            'labels': pr['labels'],
            'state': pr['state'],
            'additions': pr['additions'],
            'deletions': pr['deletions'],
            'files_changed': pr['changed_files_count'],
            'languages': list(pr['language_stats'].keys()) if pr['language_stats'] else []
        }
        
        if pr['description'] and len(pr['description']) < 500:
            summary['description'] = pr['description'][:300]
        
        pr_summary.append(summary)
    
    return pr_summary

if llm and developer_stats:
    print(f"🔧 Creating analysis chains for ALL developers...")
    
    skills_chain, profile_chain = create_analysis_chains(llm)
    
    eligible_developers = [
        (name, stats) for name, stats in developer_stats.items()
        if stats['total_prs'] >= 3 and not name.endswith('[bot]') and 'bot' not in name.lower()
    ]
    
    eligible_developers.sort(key=lambda x: x[1]['total_prs'], reverse=True)
    
    print(f"✅ Chains created successfully")
    print(f"👥 Found {len(eligible_developers)} eligible developers")
    print(f"📊 Will process developers with 3+ PRs (excluding bots)")
    
elif llm:
    print("⚠️  LLM configured but no developers found")
    print("Please check that PR data was loaded correctly")
else:
    print("⚠️  LLM not configured - please run the LLM configuration cell first")
    print("Set up your Gemini API key to continue")

🔧 Creating analysis chains for ALL developers...
✅ Chains created successfully
👥 Found 36 eligible developers
📊 Will process developers with 3+ PRs (excluding bots)


In [36]:
def generate_developer_profile(developer_name, developer_data, pr_summary):
    try:
        skills_analysis_input = {
            "developer_name": developer_name,
            "pr_data": json.dumps(pr_summary, indent=2)
        }
        
        print(f"  🔍 Analyzing skills for {developer_name}...")
        skills_analysis_result = skills_chain.run(skills_analysis_input)
        
        profile_input = {
            "developer_name": developer_name,
            "skills_analysis": skills_analysis_result,
            "pr_count": developer_data['total_prs'],
            "repo_list": ', '.join(developer_data['repos'])
        }
        
        print(f"  🤖 Generating profile for {developer_name}...")
        profile_result = profile_chain.run(profile_input)
        
        try:
            parsed_profile = profile_parser.parse(profile_result)
            profile_dict = parsed_profile.dict()
        except Exception as parse_error:
            print(f"  ⚠️  Profile parsing failed for {developer_name}: {parse_error}")
            profile_dict = {
                "developer_name": developer_name,
                "raw_response": profile_result,
                "parsing_error": str(parse_error)
            }
        
        return {
            "success": True,
            "profile": profile_dict,
            "skills_analysis": skills_analysis_result,
            "raw_profile": profile_result
        }
        
    except Exception as e:
        print(f"  ❌ Error generating profile for {developer_name}: {e}")
        return {
            "success": False,
            "error": str(e),
            "developer": developer_name
        }

In [37]:
import time

def load_checkpoint(checkpoint_file="profile_generation_checkpoint.json"):
    """Load the last checkpoint to resume processing"""
    if os.path.exists(checkpoint_file):
        try:
            with open(checkpoint_file, 'r', encoding='utf-8') as f:
                checkpoint = json.load(f)
            return checkpoint
        except Exception as e:
            print(f"⚠️  Error loading checkpoint: {e}")
            return None
    return None

def save_checkpoint(processed_developers, checkpoint_file="profile_generation_checkpoint.json"):
    """Save current progress checkpoint"""
    checkpoint = {
        "processed_developers": processed_developers,
        "last_updated": datetime.now().isoformat(),
        "total_processed": len(processed_developers)
    }
    
    try:
        with open(checkpoint_file, 'w', encoding='utf-8') as f:
            json.dump(checkpoint, f, indent=2)
        print(f"💾 Checkpoint saved: {len(processed_developers)} developers processed")
    except Exception as e:
        print(f"❌ Error saving checkpoint: {e}")

def get_remaining_developers(eligible_developers, processed_developers=None):
    """Get list of developers that haven't been processed yet"""
    if not processed_developers:
        return eligible_developers
    
    processed_names = set(processed_developers)
    remaining = [(name, stats) for name, stats in eligible_developers 
                 if name not in processed_names]
    
    return remaining

def batch_generate_profiles_with_resume(eligible_developers, batch_size=15, delay_seconds=2, 
                                      checkpoint_file="profile_generation_checkpoint.json"):
    """Generate profiles with checkpoint/resume capability"""
    
    # Load existing checkpoint
    checkpoint = load_checkpoint(checkpoint_file)
    processed_developers = checkpoint.get('processed_developers', []) if checkpoint else []
    
    # Get remaining developers to process
    remaining_developers = get_remaining_developers(eligible_developers, processed_developers)
    
    if not remaining_developers:
        print("🎉 All developers have already been processed!")
        return []
    
    print(f"📊 RESUME STATUS:")
    print(f"   ✅ Already processed: {len(processed_developers)} developers")
    print(f"   ⏳ Remaining to process: {len(remaining_developers)} developers")
    print(f"   🎯 This batch will process: {min(batch_size, len(remaining_developers))} developers")
    
    # Process only the batch size
    current_batch = remaining_developers[:batch_size]
    batch_results = []
    successful_profiles = 0
    failed_profiles = 0
    
    print(f"\n🚀 Starting batch processing ({len(current_batch)} developers)...")
    
    for i, (dev_name, dev_data) in enumerate(current_batch, 1):
        try:
            print(f"\nProcessing {i}/{len(current_batch)}: {dev_name} ({dev_data['total_prs']} PRs)")
            
            pr_summary = prepare_pr_data_for_analysis(dev_data, max_prs=15)
            result = generate_developer_profile(dev_name, dev_data, pr_summary)
            
            if result["success"]:
                batch_results.append({
                    "developer": dev_name,
                    "result": result,
                    "stats": dev_data,
                    "processed_at": datetime.now().isoformat()
                })
                
                # Add to processed list and save checkpoint immediately
                processed_developers.append(dev_name)
                save_checkpoint(processed_developers, checkpoint_file)
                
                successful_profiles += 1
                print(f"  ✅ Success - Profile generated and checkpoint saved")
            else:
                failed_profiles += 1
                print(f"  ❌ Failed: {result.get('error', 'Unknown')}")
                
        except Exception as e:
            failed_profiles += 1
            print(f"  💥 Exception: {e}")
        
        # Delay between API calls (except for the last one)
        if i < len(current_batch):
            print(f"  ⏸️  Waiting {delay_seconds} seconds...")
            time.sleep(delay_seconds)
    
    # Final summary
    total_processed_now = len(processed_developers)
    total_remaining = len(remaining_developers) - len(current_batch)
    
    print(f"\n🎉 BATCH COMPLETED!")
    print(f"✅ This batch successful: {successful_profiles}")
    print(f"❌ This batch failed: {failed_profiles}")
    print(f"📊 Total processed so far: {total_processed_now}/{len(eligible_developers)} developers")
    print(f"⏳ Still remaining: {total_remaining} developers")
    
    if total_remaining > 0:
        print(f"\n💡 To continue processing, run the next cell again!")
        print(f"   Next batch will process up to {min(batch_size, total_remaining)} more developers")
    else:
        print(f"\n🎊 ALL DEVELOPERS COMPLETED! Cleaning up checkpoint...")
        # Optionally remove checkpoint file when done
        if os.path.exists(checkpoint_file):
            os.remove(checkpoint_file)
    
    return batch_results

def export_profiles_from_checkpoint(checkpoint_file="profile_generation_checkpoint.json", 
                                  output_dir="generated_profiles"):
    """Export all profiles that have been generated so far"""
    
    # Load all existing profile files
    if not os.path.exists(output_dir):
        print(f"📁 Output directory {output_dir} doesn't exist yet")
        return []
    
    existing_files = [f for f in os.listdir(output_dir) if f.endswith('_profile.json')]
    print(f"💾 Found {len(existing_files)} existing profile files in {output_dir}/")
    
    return existing_files

def export_batch_profiles(batch_results, output_dir="Developer's Profiles"):
    """Export profiles from a single batch"""
    if not batch_results:
        return []
    
    os.makedirs(output_dir, exist_ok=True)
    exported_files = []
    
    print(f"💾 Exporting {len(batch_results)} new profiles...")
    
    for i, profile_data in enumerate(batch_results, 1):
        dev_name = profile_data["developer"]
        result = profile_data["result"]
        stats = profile_data["stats"]
        
        export_data = {
            "metadata": {
                "developer_name": dev_name,
                "generation_timestamp": profile_data["processed_at"],
                "llm_provider": "OpenRouter (Qwen 2.5)",
                "source": "GitHub PR Data Analysis",
                "total_prs_analyzed": stats['total_prs'],
                "repositories": stats['repos'],
                "pr_data_sample_size": min(15, len(stats['pr_details']))
            },
            "profile": result.get("profile"),
            "statistics": {
                "total_prs": stats['total_prs'],
                "repos_contributed": len(stats['repos']),
                "repo_list": stats['repos']
            }
        }
        
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"{dev_name}_profile_{timestamp}.json"
        filepath = os.path.join(output_dir, filename)
        
        try:
            with open(filepath, 'w', encoding='utf-8') as f:
                json.dump(export_data, f, indent=2, ensure_ascii=False)
            
            exported_files.append(filepath)
            
        except Exception as e:
            print(f"❌ Export failed for {dev_name}: {e}")
    
    print(f"💾 Exported {len(exported_files)} profile files to {output_dir}/")
    return exported_files

In [38]:
def check_progress_status():
    """Check current progress status"""
    checkpoint = load_checkpoint()
    
    if not checkpoint:
        print("📊 No checkpoint found - starting fresh")
        print(f"🎯 Total developers to process: {len(eligible_developers)}")
        return
    
    processed = len(checkpoint['processed_developers'])
    total = len(eligible_developers)
    remaining = total - processed
    percentage = (processed / total) * 100
    
    print("📊 PROGRESS STATUS")
    print("=" * 40)
    print(f"✅ Completed: {processed}/{total} developers ({percentage:.1f}%)")
    print(f"⏳ Remaining: {remaining} developers")
    print(f"📅 Last updated: {checkpoint['last_updated']}")
    
    if remaining > 0:
        print(f"\n💡 Run the next cell to continue processing")
    else:
        print(f"🎉 All developers completed!")

# Enhanced usage function
def run_profile_generation_session(batch_size=15, delay_seconds=2):
    """Run a single profile generation session"""
    
    if not (llm and developer_stats and 'eligible_developers' in globals()):
        print("❌ Please run the setup cells first (LLM configuration and developer analysis)")
        return
    
    print("🎯 STARTING PROFILE GENERATION SESSION")
    print("=" * 60)
    
    batch_results = batch_generate_profiles_with_resume(
        eligible_developers=eligible_developers,
        batch_size=batch_size,
        delay_seconds=delay_seconds
    )
    
    # Export this batch if we got results
    if batch_results:
        exported_files = export_batch_profiles(batch_results)
        print(f"💾 Exported {len(exported_files)} new profile files")
    
    # Show status of all profiles
    existing_profiles = export_profiles_from_checkpoint()
    
    return batch_results

# Check current status
print("🔍 CHECKING CURRENT PROGRESS...")
check_progress_status()

🔍 CHECKING CURRENT PROGRESS...
📊 PROGRESS STATUS
✅ Completed: 4/36 developers (11.1%)
⏳ Remaining: 32 developers
📅 Last updated: 2025-11-14T17:43:37.735921

💡 Run the next cell to continue processing


# Run Profile Generation in Batches

In [39]:

BATCH_SIZE = 40     
DELAY_SECONDS = 2      


batch_results = run_profile_generation_session(
    batch_size=BATCH_SIZE, 
    delay_seconds=DELAY_SECONDS
)


if batch_results:
    print(f"\n📋 Quick Preview of This Batch:")
    for i, profile_data in enumerate(batch_results[:3], 1):
        dev_name = profile_data["developer"]
        profile = profile_data["result"]["profile"]
        if profile:
            experience = profile.get('experience_level', 'Unknown')
            skills_count = len(profile.get('primary_skills', []))
            print(f"  {i}. {dev_name}: {experience} level, {skills_count} skills")
    
    if len(batch_results) > 3:
        print(f"  ... and {len(batch_results) - 3} more profiles generated")
else:
    print("📝 No new profiles generated in this batch")

🎯 STARTING PROFILE GENERATION SESSION
📊 RESUME STATUS:
   ✅ Already processed: 4 developers
   ⏳ Remaining to process: 32 developers
   🎯 This batch will process: 32 developers

🚀 Starting batch processing (32 developers)...

Processing 1/32: falsyvalues (16 PRs)
  🔍 Analyzing skills for falsyvalues...


> Entering new LLMChain chain...
Prompt after formatting:
Analyze the following GitHub Pull Request data for developer falsyvalues and extract their JavaScript technical skills and expertise:

PR Data:
[
  {
    "title": "unescape broken after remove semicolons action",
    "repo": "lodash",
    "labels": [
      "bug"
    ],
    "state": "closed",
    "additions": 0,
    "deletions": 0,
    "files_changed": 0,
    "languages": [],
    "description": ":+1: "
  },
  {
    "title": "Use parentheses around arrow function argument having a body with cur\u2026",
    "repo": "lodash",
    "labels": [
      "enhancement",
      "wontfix"
    ],
    "state": "closed",
    "additions": 0,
    "